In [1]:
states = [0, 1, 2, 3]
policy = [0, 0, 0, 0]  # initial policy
possible_actions = [0, 1, 2, 3]  # actions: 0=up, 1=down, 2=left, 3=right

def environment_step(state, action) -> tuple[int, int]:
    # Define the environment dynamics here
    # along with reward
    if action == 0:  # up
        if state in [2, 3]:
            state -= 2
            if state == 3:
                return state, 1  # reward for reaching state 3
        return state, 0  # no reward for other states
    if action == 1:  # down
        if state in [0, 1]:
            state += 2
            if state == 3:
                return state, 1
        if state == 3:
            return state, 1  # reward for reaching state 3
        return state, 0  # no reward for other states
    if action == 2:  # left
        if state in [1, 3]:
            state -= 1
            if state == 3:
                return state, 1
        if state == 3:
            return state, 1  # reward for reaching state 3
        return state, 0  # no reward for other states    
    if action == 3:  # right
        if state in [0, 2]:
            state += 1
            if state == 3:
                return state, 1
        if state == 3:
            return state, 1  # reward for reaching state 3
        return state, 0  # no reward for other states

## Monte Carlo Exploring Starts Algorithm
episodes = 3
q_values = [[0, 0, 0, 0] for _ in range(4)]  # Initialize Q-values for each state-action pair
nums = [[0, 0, 0, 0] for _ in range(4)]  # Count of visits for each state-action pair
returns = [[0, 0, 0, 0] for _ in range(4)]  # Cumulative returns for each state-action pair

for episode in range(episodes):
    # start from every state action pair then follow through with the policy
    for state in states:
        for action in possible_actions:
            trajectory = []
            # Take action and observe reward and next state
            next_state, reward = environment_step(state, action)
            # Follow the policy for 3 steps
            trajectory.append((state, action, reward))
            for _ in range(3):
                state = next_state
                if state == 3:  # If we reach the terminal state, break
                    break
                action = policy[next_state]
                next_state, reward = environment_step(state, action)
                trajectory.append((state, action, reward))
            # go backwards and update q_values
            G = 0
            for t in reversed(range(len(trajectory))):
                state, action, reward = trajectory[t]
                G = reward + 0.9 * G  # Update the return
                # Update the Q-value for the state-action pair
                returns[state][action] += G
                nums[state][action] += 1
                # Policy evaluation
                q_values[state][action] = returns[state][action] / nums[state][action]
                # Policy improvement
                policy[state] = q_values[state].index(max(q_values[state]))
            

print("Optimal Policy:")
for s in range(4):
    print(f"State {s}: Action {policy[s]}")
                

Optimal Policy:
State 0: Action 1
State 1: Action 1
State 2: Action 3
State 3: Action 1


In [2]:
import random

class GridEnvironment:
    """2D grid environment. Terminal when a tile's reward != 0."""
    def __init__(self, width, height, rewards=None):
        self.width = width
        self.height = height
        self.n_tiles = width * height
        self.rewards = list(rewards) if rewards is not None else [0.0] * self.n_tiles
        if len(self.rewards) != self.n_tiles:
            raise ValueError("rewards must have length width * height")

    def to_index(self, row, col):
        return row * self.width + col

    def to_coord(self, state):
        return divmod(state, self.width)

    def step(self, state, action):
        """Apply action from state and return (next_state, reward, done)."""
        row, col = self.to_coord(state)
        if action == 0:  # up
            row = max(0, row - 1)
        elif action == 1:  # down
            row = min(self.height - 1, row + 1)
        elif action == 2:  # left
            col = max(0, col - 1)
        elif action == 3:  # right
            col = min(self.width - 1, col + 1)
        next_state = self.to_index(row, col)
        reward = self.rewards[next_state]
        done = reward != 0.0
        return next_state, reward, done


class MonteCarloESAgent:
    """Monte Carlo Exploring Starts (first-visit) agent for discrete grid.

    This implementation follows the deterministic schedule: each episode iterates
    through every state-action pair as the exploring start (no random starts),
    matching the original algorithm you had in the first cell.
    """
    def __init__(self, env: GridEnvironment, gamma=0.9):
        self.env = env
        self.gamma = gamma
        self.n_actions = 4
        # initialize policy randomly for non-terminal states; terminal states have None
        self.policy = [None if self.env.rewards[s] != 0.0 else random.randrange(self.n_actions)
                       for s in range(self.env.n_tiles)]
        self.Q = [[0.0] * self.n_actions for _ in range(self.env.n_tiles)]
        self.returns_sum = [[0.0] * self.n_actions for _ in range(self.env.n_tiles)]
        self.returns_count = [[0] * self.n_actions for _ in range(self.env.n_tiles)]
        self.action_symbols = {0: '↑', 1: '↓', 2: '←', 3: '→'}

    def generate_episode_from(self, start_state, start_action, max_steps=50):
        """Generate an episode starting from given state-action (exploring start).
        Returns list of (state, action, reward) tuples.
        """
        episode = []
        state = start_state
        action = start_action
        next_state, reward, done = self.env.step(state, action)
        episode.append((state, action, reward))
        state = next_state
        steps = 0
        while not done and steps < max_steps:
            action = self.policy[state]
            if action is None:
                break
            next_state, reward, done = self.env.step(state, action)
            episode.append((state, action, reward))
            state = next_state
            steps += 1
        return episode

    def learn(self, n_passes=10, max_steps_per_episode=50):
        """Run the deterministic ES schedule for a number of full passes.

        For each pass we iterate every state s and every action a (the exploring
        start), generate an episode starting from (s,a) following current
        policy afterwards, then perform first-visit MC updates and greedy
        policy improvement.
        """
        for _ in range(n_passes):
            for start_state in range(self.env.n_tiles):
                for start_action in range(self.n_actions):
                    # Generate episode starting from this state-action pair
                    episode = self.generate_episode_from(start_state, start_action, max_steps_per_episode)

                    G = 0.0
                    visited = set()
                    # compute returns in reverse (first-visit)
                    for t in range(len(episode) - 1, -1, -1):
                        s, a, r = episode[t]
                        G = self.gamma * G + r
                        if (s, a) not in visited:
                            visited.add((s, a))
                            self.returns_sum[s][a] += G
                            self.returns_count[s][a] += 1
                            self.Q[s][a] = self.returns_sum[s][a] / self.returns_count[s][a]
                            # policy improvement (greedy) for non-terminal states
                            if self.env.rewards[s] == 0.0:
                                best_a = max(range(self.n_actions), key=lambda x: self.Q[s][x])
                                self.policy[s] = best_a

    def policy_grid(self):
        grid = []
        for row in range(self.env.height):
            row_symbols = []
            for col in range(self.env.width):
                s = self.env.to_index(row, col)
                if self.env.rewards[s] != 0.0:
                    row_symbols.append('T')
                else:
                    a = self.policy[s]
                    row_symbols.append(self.action_symbols[a] if a is not None else '?')
            grid.append(row_symbols)
        return grid

    def values_grid(self):
        grid = []
        for row in range(self.env.height):
            row_vals = []
            for col in range(self.env.width):
                s = self.env.to_index(row, col)
                if self.env.rewards[s] != 0.0:
                    row_vals.append(self.env.rewards[s])
                else:
                    row_vals.append(round(max(self.Q[s]), 2))
            grid.append(row_vals)
        return grid

    def print_policy(self):
        print("Learned policy (T = terminal):")
        for row in self.policy_grid():
            print(" ".join(row))
        print("\nState values / terminal rewards:")
        for row in self.values_grid():
            print(" ".join(f"{v:.2f}" for v in row))


In [ ]:
# Example usage: Monte Carlo Exploring Starts on a 4x4 grid
rewards = [0.0, 0.0, 0.0, 0.0,
           0.0, 0.0, 0.0, 1.0,
           0.0, 0.0, -1.0, 0.0,
           0.0, 0.0, 0.0, 0.0]

env = GridEnvironment(width=4, height=4, rewards=rewards)
agent = MonteCarloESAgent(env, gamma=0.9)

print("Learning...")
agent.learn(n_passes=10, max_steps_per_episode=50)
print("Done. Learned policy:")
agent.print_policy()


Learning...
Done. Learned policy:
Learned policy (T = terminal):
↓ ↓ → ↓
↓ → → T
→ ↑ T ↑
→ → → ↑

State values / terminal rewards:
0.53 0.77 0.88 1.00
0.64 0.90 1.00 1.00
0.70 0.74 -1.00 1.00
0.63 0.72 0.81 0.90


In [8]:
# Example usage: Monte Carlo Exploring Starts on a 4x4 grid
rewards = [0.0, 0.0, 0.0, 0.0,
           0.0, -1.0, 0.0, 0.0,
           0.0, 0.0, -1.0, 0.0,
           1.0, 0.0, 0.0, 0.0]

env = GridEnvironment(width=4, height=4, rewards=rewards)
agent = MonteCarloESAgent(env, gamma=0.9)

print("Learning...")
agent.learn(n_passes=70, max_steps_per_episode=50)
print("Done. Learned policy:")
agent.print_policy()


Learning...
Done. Learned policy:
Learned policy (T = terminal):
↓ ← ← ←
↓ T → ↓
↓ ← T ↓
T ← ← ←

State values / terminal rewards:
0.81 0.73 0.65 0.59
0.90 -1.00 0.59 0.65
1.00 0.90 -1.00 0.73
1.00 1.00 0.90 0.81


In [12]:
import random

states = [0, 1, 2, 3]
policy = [0, 0, 0, 0]  # initial policy
possible_actions = [0, 1, 2, 3]  # actions: 0=up, 1=down, 2=left, 3=right

def environment_step(state, action) -> tuple[int, int]:
    # Define the environment dynamics here
    # along with reward
    if action == 0:  # up
        if state in [2, 3]:
            state -= 2
            if state == 3:
                return state, 1  # reward for reaching state 3
        return state, 0  # no reward for other states
    if action == 1:  # down
        if state in [0, 1]:
            state += 2
            if state == 3:
                return state, 1
        if state == 3:
            return state, 1  # reward for reaching state 3
        return state, 0  # no reward for other states
    if action == 2:  # left
        if state in [1, 3]:
            state -= 1
            if state == 3:
                return state, 1
        if state == 3:
            return state, 1  # reward for reaching state 3
        return state, 0  # no reward for other states    
    if action == 3:  # right
        if state in [0, 2]:
            state += 1
            if state == 3:
                return state, 1
        if state == 3:
            return state, 1  # reward for reaching state 3
        return state, 0  # no reward for other states

## Monte Carlo Epsilon Greedy Algorithm
epsilon = 0.5
episodes = 10
q_values = [[0, 0, 0, 0] for _ in range(4)]  # Initialize Q-values for each state-action pair
nums = [[0, 0, 0, 0] for _ in range(4)]  # Count of visits for each state-action pair
returns = [[0, 0, 0, 0] for _ in range(4)]  # Cumulative returns for each state-action pair

for episode in range(episodes):
    for state in states:
        action = random.choice(possible_actions)  # Start with a random action
        trajectory = []
        # Take action and observe reward and next state
        next_state, reward = environment_step(state, action)
        # Follow the policy for 3 steps
        trajectory.append((state, action, reward))
        for _ in range(3):
            state = next_state
            if state == 3:  # If we reach the terminal state, break
                break
            # Epsilon-greedy action selection
            if random.random() < epsilon:
                action = random.choice(possible_actions)
            else:
                action = policy[next_state]
            next_state, reward = environment_step(state, action)
            trajectory.append((state, action, reward))
        # go backwards and update q_values
        G = 0
        for t in reversed(range(len(trajectory))):
            state, action, reward = trajectory[t]
            G = reward + 0.9 * G  # Update the return
            # Update the Q-value for the state-action pair
            returns[state][action] += G
            nums[state][action] += 1
            # Policy evaluation
            q_values[state][action] = returns[state][action] / nums[state][action]
            # Policy improvement
            policy[state] = q_values[state].index(max(q_values[state]))
            

print("Optimal Policy:")
for s in range(4):
    print(f"State {s}: Action {policy[s]}")
                

Optimal Policy:
State 0: Action 3
State 1: Action 1
State 2: Action 3
State 3: Action 1


In [15]:
class MonteCarloEGAgent:
    """Monte Carlo Epsilon-Greedy (first-visit) agent for discrete grid.

    This implementation follows the deterministic schedule: each episode iterates
    through every state-action pair as the exploring start (no random starts),
    matching the original algorithm you had in the first cell.
    """
    def __init__(self, env: GridEnvironment, gamma=0.9, epsilon=0.1):
        self.env = env
        self.gamma = gamma
        self.epsilon = epsilon
        self.n_actions = 4
        # initialize policy randomly for non-terminal states; terminal states have None
        self.policy = [None if self.env.rewards[s] != 0.0 else random.randrange(self.n_actions)
                       for s in range(self.env.n_tiles)]
        self.Q = [[0.0] * self.n_actions for _ in range(self.env.n_tiles)]
        self.returns_sum = [[0.0] * self.n_actions for _ in range(self.env.n_tiles)]
        self.returns_count = [[0] * self.n_actions for _ in range(self.env.n_tiles)]
        self.action_symbols = {0: '↑', 1: '↓', 2: '←', 3: '→'}

    def generate_episode_from(self, start_state, start_action, max_steps=50):
        """Generate an episode starting from given state-action (exploring start).
        Returns list of (state, action, reward) tuples.
        """
        episode = []
        state = start_state
        action = start_action
        next_state, reward, done = self.env.step(state, action)
        episode.append((state, action, reward))
        state = next_state
        steps = 0
        while not done and steps < max_steps:
            if random.random() < self.epsilon:
                action = random.choice(possible_actions)
            else:
                action = self.policy[next_state]
            if action is None:
                break
            next_state, reward, done = self.env.step(state, action)
            episode.append((state, action, reward))
            state = next_state
            steps += 1
        return episode

    def learn(self, n_passes=10, max_steps_per_episode=50):
        """Run the stochastic EG schedule for a number of full passes.

        For each pass we iterate every state s and every action a (the exploring
        start), generate an episode starting from (s,a) following current
        policy afterwards, then perform first-visit MC updates and greedy
        policy improvement.
        """
        for _ in range(n_passes):
            for start_state in range(self.env.n_tiles):
                start_action = random.randrange(self.n_actions)  # Random action for epsilon-greedy
                # Generate episode starting from this state-action pair
                episode = self.generate_episode_from(start_state, start_action, max_steps_per_episode)

                G = 0.0
                visited = set()
                # compute returns in reverse (first-visit)
                for t in range(len(episode) - 1, -1, -1):
                    s, a, r = episode[t]
                    G = self.gamma * G + r
                    if (s, a) not in visited:
                        visited.add((s, a))
                        self.returns_sum[s][a] += G
                        self.returns_count[s][a] += 1
                        self.Q[s][a] = self.returns_sum[s][a] / self.returns_count[s][a]
                        # policy improvement (greedy) for non-terminal states
                        if self.env.rewards[s] == 0.0:
                            best_a = max(range(self.n_actions), key=lambda x: self.Q[s][x])
                            self.policy[s] = best_a

    def policy_grid(self):
        grid = []
        for row in range(self.env.height):
            row_symbols = []
            for col in range(self.env.width):
                s = self.env.to_index(row, col)
                if self.env.rewards[s] != 0.0:
                    row_symbols.append('T')
                else:
                    a = self.policy[s]
                    row_symbols.append(self.action_symbols[a] if a is not None else '?')
            grid.append(row_symbols)
        return grid

    def values_grid(self):
        grid = []
        for row in range(self.env.height):
            row_vals = []
            for col in range(self.env.width):
                s = self.env.to_index(row, col)
                if self.env.rewards[s] != 0.0:
                    row_vals.append(self.env.rewards[s])
                else:
                    row_vals.append(round(max(self.Q[s]), 2))
            grid.append(row_vals)
        return grid

    def print_policy(self):
        print("Learned policy (T = terminal):")
        for row in self.policy_grid():
            print(" ".join(row))
        print("\nState values / terminal rewards:")
        for row in self.values_grid():
            print(" ".join(f"{v:.2f}" for v in row))


In [16]:
# Example usage: Monte Carlo Exploring Starts on a 4x4 grid
rewards = [0.0, 0.0, 0.0, 0.0,
           0.0, -1.0, 0.0, 0.0,
           0.0, 0.0, -1.0, 0.0,
           1.0, 0.0, 0.0, 0.0]

env = GridEnvironment(width=4, height=4, rewards=rewards)
agent = MonteCarloEGAgent(env, gamma=0.9, epsilon=0.5)

print("Learning...")
agent.learn(n_passes=30, max_steps_per_episode=50)
print("Done. Learned policy:")
agent.print_policy()


Learning...
Done. Learned policy:
Learned policy (T = terminal):
↓ ← ← ↓
↓ T → ↓
↓ ↓ T ↓
T ← ← ←

State values / terminal rewards:
0.29 0.21 0.01 0.08
0.80 -1.00 -0.10 0.09
1.00 0.69 -1.00 0.32
1.00 1.00 0.75 0.37
